# Notebook 1 — Baseline CNN on CIFAR-10
**Deep Learning | Computer Vision | 2025**

Build a minimal 2-block CNN on CIFAR-10 with **no regularisation** to establish the performance baseline.

| | |
|---|---|
| Dataset | CIFAR-10 — 50,000 train / 10,000 test / 10 classes / 32×32 RGB |
| Architecture | 2-block CNN |
| Target accuracy | **~65%** |

> ✅ Open in **Google Colab** → Runtime → Change runtime type → **GPU**


## 1. Imports & Setup

In [ ]:
from time import time
import os, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import SGD, RMSprop, Adagrad, Adadelta, Adam, Adamax, Nadam
from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_score, recall_score, f1_score)
warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {bool(tf.config.list_physical_devices('GPU'))}")

## 2. Load & Explore CIFAR-10

In [ ]:
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()
print(f"Train: {train_images.shape}  |  Test: {test_images.shape}")
print(f"Pixel range: [{train_images.min()}, {train_images.max()}]")

In [ ]:
# First 25 training images
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(CLASS_NAMES[train_labels[i][0]], fontsize=9)
plt.suptitle('CIFAR-10 — Sample Images', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('01_cifar10_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Class distribution
counts = np.bincount(train_labels.flatten())
fig, ax = plt.subplots(figsize=(11,4))
bars = ax.bar(CLASS_NAMES, counts, color=plt.cm.tab10(np.linspace(0,1,10)))
for b, c in zip(bars, counts):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+40,
            str(c), ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0,6500)
ax.set_title('Training Set — Class Distribution (Perfectly Balanced: 5,000/class)',
             fontweight='bold')
ax.set_ylabel('Images per Class')
plt.xticks(rotation=30); plt.tight_layout()
plt.savefig('01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing — Normalise Pixels to [0, 1]

In [ ]:
train_images_n = train_images.astype('float32') / 255.0
test_images_n  = test_images.astype('float32')  / 255.0

# One-hot encode labels
train_labels_enc = keras.utils.to_categorical(train_labels, 10)
test_labels_enc  = keras.utils.to_categorical(test_labels,  10)
y_true = test_labels.flatten()

print(f"Pixel range: [{train_images_n.min():.2f}, {train_images_n.max():.2f}]")
print(f"train_labels_enc shape: {train_labels_enc.shape}")

## 4. Training Helper Function

A reusable function that compiles, trains, plots curves, evaluates,
and prints Accuracy / Precision / Recall / F1-Score.


In [ ]:
def train_and_evaluate(model, train_imgs, train_lbls, test_imgs, test_lbls,
                        epochs=10, batch_size=32,
                        optimizer=Adam(learning_rate=0.001),
                        label='Model', callbacks=None):
    """
    Compile, train, plot training curves and evaluate a Keras model.

    Returns: (history, test_accuracy, y_pred)
    """
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    print(model.summary())

    cb = callbacks or [EarlyStopping(monitor='val_loss', patience=5,
                                     restore_best_weights=True)]
    s = time()
    history = model.fit(train_imgs, train_lbls,
                        epochs=epochs, batch_size=batch_size,
                        validation_split=0.1, verbose=1, callbacks=cb)
    print(f'Training time: {round(time()-s, 2)}s')

    # ── Training curves ──────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
    fig.suptitle(f'{label} — Training History', fontsize=13, fontweight='bold')
    ax1.plot(history.history['accuracy'],     label='Train', lw=2)
    ax1.plot(history.history['val_accuracy'], label='Val',   lw=2, ls='--')
    ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(history.history['loss'],     label='Train', lw=2)
    ax2.plot(history.history['val_loss'], label='Val',   lw=2, ls='--')
    ax2.set_title('Loss'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    fname = label.replace(' ','_').replace('/','_')
    plt.savefig(f'{fname}_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Evaluation ───────────────────────────────────────────────
    test_loss, test_acc = model.evaluate(test_imgs, test_lbls, verbose=0)
    y_pred_proba = model.predict(test_imgs, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_t    = np.argmax(test_lbls, axis=1) if test_lbls.ndim>1 else test_lbls.flatten()

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"  Test Accuracy  : {test_acc*100:.2f}%")
    print(f"  Test Loss      : {test_loss:.4f}")
    print(f"  Precision      : {precision_score(y_t,y_pred,average='weighted'):.4f}")
    print(f"  Recall         : {recall_score(y_t,y_pred,average='weighted'):.4f}")
    print(f"  F1-Score       : {f1_score(y_t,y_pred,average='weighted'):.4f}")
    print(f"{'='*55}")
    return history, test_acc, y_pred

## 5. Baseline CNN Architecture

In [ ]:
def build_baseline_cnn():
    """
    Minimal 2-block CNN — no BatchNorm, no Dropout, no augmentation.
    Mirrors the baseline from the project specification.
    """
    model = models.Sequential(name='Baseline_CNN')
    # Block 1
    model.add(layers.Conv2D(64,(3,3), padding='same', activation='relu', input_shape=(32,32,3)))
    model.add(layers.MaxPooling2D((2,2), padding='same'))
    # Block 2
    model.add(layers.Conv2D(64,(3,3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D((2,2), padding='same'))
    # Classifier
    model.add(layers.Flatten())
    model.add(layers.Dense(120, activation='relu'))
    model.add(layers.Dense(10, activation='softmax'))
    return model

baseline_model = build_baseline_cnn()
baseline_model.summary()
print(f"\nTotal parameters: {baseline_model.count_params():,}")

## 6. Train Baseline CNN

In [ ]:
history_base, acc_base, y_pred_base = train_and_evaluate(
    baseline_model,
    train_images_n, train_labels_enc,
    test_images_n,  test_labels_enc,
    epochs=10, batch_size=32,
    optimizer=Adam(learning_rate=0.001),
    label='Baseline CNN',
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

## 7. Classification Report & Confusion Matrix

In [ ]:
print("Per-Class Classification Report:")
print(classification_report(y_true, y_pred_base, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred_base)
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Baseline CNN — Confusion Matrix', fontsize=13, fontweight='bold')
ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig('01_baseline_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Prediction grid with confidence scores
y_proba = baseline_model.predict(test_images_n, verbose=0)
y_conf  = np.max(y_proba, axis=1)
fig = plt.figure(figsize=(13,13))
fig.suptitle('Baseline CNN — Sample Predictions (Green=Correct, Red=Wrong)',
             fontsize=13, fontweight='bold')
idxs = np.random.choice(len(y_true), 16, replace=False)
for i, idx in enumerate(idxs):
    ax = fig.add_subplot(4,4,i+1)
    ax.imshow(test_images[idx])
    pred = CLASS_NAMES[y_pred_base[idx]]; true = CLASS_NAMES[y_true[idx]]
    ax.set_title(f"T: {true}\nP: {pred}\n{y_conf[idx]:.2f}",
                 fontsize=8, color='green' if pred==true else 'red')
    ax.axis('off')
plt.tight_layout()
plt.savefig('01_baseline_sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary

| Metric | Value |
|---|---|
| **Test Accuracy** | **~65%** |
| Parameters | ~470K |
| Epochs | 10, Batch 32 |
| Observation | Overfits after epoch 5 — val_loss rises |

**Bottlenecks:**
- No BatchNormalization → unstable gradients
- No Dropout → overfits training data
- No data augmentation → poor generalisation
- Only 2 conv blocks → shallow feature hierarchy

➡️ **Notebook 2** applies BatchNorm, Dropout, Augmentation, deeper blocks → **~82%**
